# 📊 Análise e Visualização de Resultados

Este notebook mostra como analisar e visualizar resultados de benchmarks de forma aprofundada.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Configurar matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Carregando Dados de Múltiplas Execuções

In [ ]:
def load_metrics_snapshot(filepath):
    """Carrega snapshot de métricas e converte para DataFrame."""
    with open(filepath) as f:
        data = json.load(f)
    
    # Converter formato dict para list se necessário
    if isinstance(data, dict):
        rows = []
        for method_name, metrics in data.items():
            if isinstance(metrics, dict):
                row = {"method": method_name}
                row.update(metrics)
                rows.append(row)
        data = rows
    
    return pd.DataFrame(data)

# Exemplo: Carregar resultados
snapshot_path = Path("./artifacts/metrics/latest_preview.json")
if snapshot_path.exists():
    df = load_metrics_snapshot(snapshot_path)
    print("Métricas carregadas:")
    print(df.head())
else:
    print(f"⚠️  Arquivo não encontrado: {snapshot_path}")
    print("Execute alguns benchmarks primeiro!")

## 2. Comparação de Qualidade (PSNR, SSIM, LPIPS)

In [ ]:
# Comparar qualidade entre métodos
if 'df' in locals() and len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PSNR
    if 'psnr' in df.columns:
        df.set_index('method')['psnr'].plot(kind='bar', ax=axes[0], color='skyblue')
        axes[0].set_title('PSNR (Maior é Melhor)', fontsize=12, fontweight='bold')
        axes[0].axhline(y=31.01, color='red', linestyle='--', label='NeRF Paper Baseline')
        axes[0].legend()
        axes[0].set_ylabel('PSNR (dB)')
        axes[0].tick_params(axis='x', rotation=45)
    
    # SSIM
    if 'ssim' in df.columns:
        df.set_index('method')['ssim'].plot(kind='bar', ax=axes[1], color='lightgreen')
        axes[1].set_title('SSIM (Maior é Melhor)', fontsize=12, fontweight='bold')
        axes[1].axhline(y=0.947, color='red', linestyle='--', label='NeRF Paper Baseline')
        axes[1].legend()
        axes[1].set_ylabel('SSIM')
        axes[1].tick_params(axis='x', rotation=45)
    
    # LPIPS
    if 'lpips' in df.columns:
        df.set_index('method')['lpips'].plot(kind='bar', ax=axes[2], color='coral')
        axes[2].set_title('LPIPS (Menor é Melhor)', fontsize=12, fontweight='bold')
        axes[2].axhline(y=0.082, color='red', linestyle='--', label='NeRF Paper Baseline')
        axes[2].legend()
        axes[2].set_ylabel('LPIPS')
        axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado disponível para plotar.")

## 3. Comparação de Desempenho (FPS e Tempos)

In [ ]:
if 'df' in locals() and len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # FPS
    if 'fps' in df.columns:
        df.set_index('method')['fps'].plot(kind='bar', ax=axes[0], color='gold')
        axes[0].set_title('FPS (Maior é Melhor)', fontsize=12, fontweight='bold')
        axes[0].axhline(y=30, color='green', linestyle='--', label='Tempo Real (30 FPS)')
        axes[0].legend()
        axes[0].set_ylabel('FPS')
        axes[0].tick_params(axis='x', rotation=45)
    
    # Tempo de Treino
    if 'train_seconds' in df.columns:
        (df.set_index('method')['train_seconds'] / 60).plot(kind='bar', ax=axes[1], color='orange')
        axes[1].set_title('Tempo de Treino (Menor é Melhor)', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Minutos')
        axes[1].tick_params(axis='x', rotation=45)
    
    # Tempo de Inferência
    if 'inference_seconds' in df.columns:
        (df.set_index('method')['inference_seconds'] / 60).plot(kind='bar', ax=axes[2], color='purple')
        axes[2].set_title('Tempo de Inferência (Menor é Melhor)', fontsize=12, fontweight='bold')
        axes[2].set_ylabel('Minutos')
        axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado disponível para plotar.")

## 4. Trade-off Qualidade vs Velocidade

In [ ]:
if 'df' in locals() and len(df) > 0 and 'psnr' in df.columns and 'fps' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Scatter plot: PSNR vs FPS
    scatter = ax.scatter(df['fps'], df['psnr'], s=200, alpha=0.6, 
                        c=range(len(df)), cmap='viridis')
    
    # Anotações com nomes dos métodos
    for idx, row in df.iterrows():
        ax.annotate(row['method'], 
                   (row['fps'], row['psnr']),
                   xytext=(5, 5), 
                   textcoords='offset points',
                   fontsize=9)
    
    ax.set_xlabel('FPS (Frame Rate)', fontsize=12, fontweight='bold')
    ax.set_ylabel('PSNR (dB)', fontsize=12, fontweight='bold')
    ax.set_title('Trade-off: Qualidade vs Velocidade', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Zonas de interesse
    ax.fill_between([0, 30], 30, 35, alpha=0.1, color='red', label='Baixa FPS')
    ax.fill_between([30, ax.get_xlim()[1]], 30, 35, alpha=0.1, color='green', label='Tempo Real')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Dados insuficientes para plotar trade-off.")

## 5. Tabela de Ranking

In [ ]:
if 'df' in locals() and len(df) > 0:
    print("\n📊 RANKING DE MÉTODOS\n")
    
    # Calcular scores
    ranking = pd.DataFrame()
    ranking['método'] = df['method']
    
    if 'psnr' in df.columns:
        ranking['PSNR (dB)'] = df['psnr'].round(2)
        ranking['PSNR Rank'] = df['psnr'].rank(ascending=False).astype(int)
    
    if 'ssim' in df.columns:
        ranking['SSIM'] = df['ssim'].round(4)
        ranking['SSIM Rank'] = df['ssim'].rank(ascending=False).astype(int)
    
    if 'lpips' in df.columns:
        ranking['LPIPS'] = df['lpips'].round(4)
        ranking['LPIPS Rank'] = df['lpips'].rank(ascending=True).astype(int)  # Menor é melhor
    
    if 'fps' in df.columns:
        ranking['FPS'] = df['fps'].round(2)
        ranking['FPS Rank'] = df['fps'].rank(ascending=False).astype(int)
    
    # Calcular score geral
    rank_cols = [col for col in ranking.columns if col.endswith('Rank')]
    if rank_cols:
        ranking['Score Geral'] = ranking[rank_cols].mean(axis=1).round(2)
        ranking['Rank Geral'] = ranking['Score Geral'].rank().astype(int)
    
    print(ranking.to_string(index=False))
else:
    print("Nenhum dado disponível para ranking.")

## 6. Análise de Variância (Múltiplas Cenas)

In [ ]:
def analyze_across_scenes(results_dir):
    """Analisa resultados em múltiplas cenas."""
    results_dir = Path(results_dir)
    
    all_data = []
    
    for json_file in results_dir.glob("*.json"):
        with open(json_file) as f:
            data = json.load(f)
        
        if isinstance(data, dict):
            for method, metrics in data.items():
                row = {"scene": json_file.stem, "method": method}
                row.update(metrics)
                all_data.append(row)
        elif isinstance(data, list):
            for metrics in data:
                row = {"scene": json_file.stem}
                row.update(metrics)
                all_data.append(row)
    
    if not all_data:
        return None
    
    return pd.DataFrame(all_data)

# Tentar carregar dados de múltiplas cenas
results_dir = Path("./artifacts/metrics")
if results_dir.exists():
    scene_df = analyze_across_scenes(results_dir)
    
    if scene_df is not None and len(scene_df) > 0:
        print("Análise por Cena:")
        print(scene_df.groupby('method')['psnr'].agg(['mean', 'std', 'min', 'max']).round(3))
    else:
        print("Sem dados de múltiplas cenas ainda. Execute benchmarks em diferentes datasets!")
else:
    print("Diretório de resultados não encontrado.")

## 7. Comparação com Paper Baselines

In [ ]:
# Carregar baselines dos papers
baselines_path = Path("./configs/paper_baselines.json")

if baselines_path.exists():
    with open(baselines_path) as f:
        baselines = json.load(f)
    
    print("\n📚 Baselines dos Papers\n")
    print(f"{'Método':<20} {'Paper':<30} {'PSNR':<10} {'SSIM':<10}")
    print("="*70)
    
    for method, data in baselines.items():
        for scene_data in (data.get('scenes') or [data]):
            psnr = scene_data.get('psnr', 'N/A')
            ssim = scene_data.get('ssim', 'N/A')
            paper = data.get('paper', 'Unknown')
            print(f"{method:<20} {paper:<30} {str(psnr):<10} {str(ssim):<10}")
else:
    print("Arquivo de baselines não encontrado.")

## 8. Exportar Resultados

In [ ]:
def export_results(df, output_dir="./artifacts/analysis"):
    """Exporta resultados em múltiplos formatos."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # CSV
    csv_path = output_dir / "results.csv"
    df.to_csv(csv_path, index=False)
    print(f"✓ CSV exportado: {csv_path}")
    
    # JSON
    json_path = output_dir / "results.json"
    df.to_json(json_path, orient='records', indent=2)
    print(f"✓ JSON exportado: {json_path}")
    
    # Excel (se disponível)
    try:
        excel_path = output_dir / "results.xlsx"
        df.to_excel(excel_path, index=False)
        print(f"✓ Excel exportado: {excel_path}")
    except ImportError:
        print("⚠️  openpyxl não instalado. Pule Excel.")

if 'df' in locals() and len(df) > 0:
    export_results(df)
else:
    print("Nenhum dado para exportar.")

## 🎯 Dicas de Análise

1. **Interprete Cada Métrica**
   - PSNR: Qualidade geral (quanto maior, melhor)
   - SSIM: Similaridade estrutural (quanto maior, melhor)
   - LPIPS: Qualidade perceptual (quanto menor, melhor)
   - FPS: Taxa de renderização (maior = tempo real)

2. **Considere o Trade-off**
   - Nenhum método é perfeito em todos os aspectos
   - NeRF é bom em qualidade, Gaussian Splatting em velocidade

3. **Compare com Papers**
   - Use baselines como referência
   - Pequenas diferenças (<10%) podem ser normais

4. **Variância Entre Cenas**
   - Alguns métodos são melhores para certos tipos de objetos
   - Teste em múltiplas cenas para conclusões robustas

5. **Reprodutibilidade**
   - Use `--preset full` para resultados mais confiáveis
   - Seed está fixado para reprodutibilidade